# Process any H5AD file with HRA Workflows Runner

We exemplify usage with the GTEx dataset at https://storage.googleapis.com/adult-gtex/single-cell/v9/snrna-seq-data/GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad. 

# Install and import libraries

In [27]:
%pip install nbformat anndata pandas

import os
import anndata
import pandas as pd

Note: you may need to restart the kernel to use updated packages.


# Set up Docker and WSL

Follow these steps to properly set up Docker and WSL:
1. Check if the docker group exists (it usually does):
```bash 
grep docker /etc/group
```

2. Add your WSL user to the docker group:
```bash
sudo usermod -aG docker $USER
```

3. Apply the group change: You must restart your shell or run:
```bash
newgrp docker
```
4. Additionally, in PowerShell, run:
```bash
wsl --shutdown
```

5. Test Docker access: Run this to verify that Docker works without needing sudo:'
```bash
docker ps
```

If you see a list of running containers (or an empty list with headers), you're good!


# Run `hra-worfklows-runner-setup.ipynb`

In [28]:
# run it
%run hra-worfklows-runner-setup.ipynb

Note: you may need to restart the kernel to use updated packages.


# Get GTEx dataset

In [29]:

# Make sure the data folder is present
folder_path = "data"
file_name = 'GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad'

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created.")
else:
    print(f"Folder '{folder_path}' already exists.")

# Define the path to the file. 
file_path = f'{folder_path}/{file_name}'

# Check if the file exists
if not os.path.exists(file_path):
    # If the file doesn't exist, run the curl command
    !curl -L https://storage.googleapis.com/adult-gtex/single-cell/v9/snrna-seq-data/GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad -o {file_path}
    print(f"File downloaded and saved at {file_path}")
else:
    print(f"File already exists at {file_path}")


Folder 'data' already exists.
File already exists at data/GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad


In [30]:
INPUT_H5AD="data/GTEx_8_tissues_snRNAseq_atlas_071421.public_obs.h5ad"

In [31]:
data = anndata.read_h5ad(INPUT_H5AD)
data

AnnData object with n_obs × n_vars = 209126 × 17695
    obs: 'n_genes', 'fpr', 'tissue', 'prep', 'individual', 'nGenes', 'nUMIs', 'PercentMito', 'PercentRibo', 'Age_bin', 'Sex', 'Sample ID', 'Participant ID', 'Sample ID short', 'RIN score from PAXgene tissue Aliquot', 'RIN score from Frozen tissue Aliquot', 'Autolysis Score', 'Sample Ischemic Time (mins)', 'Tissue Site Detail', 'scrublet', 'scrublet_score', 'barcode', 'batch', 'n_counts', 'tissue-individual-prep', 'Broad cell type', 'Granular cell type', 'introns', 'junctions', 'exons', 'sense', 'antisense', 'intergenic', 'batch-barcode', 'exon_ratio', 'intron_ratio', 'junction_ratio', 'log10_nUMIs', 'leiden', 'leiden_tissue', 'Tissue composition', 'Cell types level 2', 'Cell types level 3', 'Broad cell type numbers', 'Broad cell type (numbers)', 'Tissue', 'channel'
    var: 'gene_ids', 'Chromosome', 'Source', 'Start', 'End', 'Strand', 'gene_name', 'gene_source', 'gene_biotype', 'gene_length', 'gene_coding_length', 'Approved symbol', '

# Prepare `hra-workflows-runner` run

In [32]:
# Taken from https://github.com/hubmapconsortium/hra-workflows-runner/blob/main/src%2Fgtex%2Fdownloader.js#L24-L39
ORGAN_MAPPING = {
    "bladder": "UBERON:0001255",
    "blood": "UBERON:0000178",
    "bone_marrow": "UBERON:0002371",
    "eye": "UBERON:0000970",
    "heart": "UBERON:0000948",
    "large_intestine": "UBERON:0000059",
    "liver": "UBERON:0002107",
    "lung": "UBERON:0002048",
    "lymph_node": "UBERON:0000029",  # or mesenteric lymph node (UBERON:0002509)?
    "mammary": "UBERON:0001911",
    "pancreas": "UBERON:0001264",
    "prostate": "UBERON:0002367",
    "skin": "UBERON:0002097",
    "small_intestine": "UBERON:0002108",
    "spleen": "UBERON:0002106",
    "thymus": "UBERON:0002370",
    "trachea": "UBERON:0003126",
    "uterus": "UBERON:0000995",
    "vasculature": "UBERON:0004537",
    "breast": "UBERON:0001911",
    "esophagus mucosa": "UBERON:0002469",
    "esophagus muscularis": "UBERON:0004648",
    "skeletal muscle": "UBERON:0001134",
}

In [33]:
queryLayersKey = "counts"

samples = data.obs['Sample ID'].unique()
for sample in samples:
    dataset_id = f"urn:gtex:{sample}"
    subset_dir=f"data/gtex/GTEX-{sample}"
    subset_h5ad = f"{subset_dir}/data.h5ad"
    tissue = subset.obs['Tissue'].values[0].lower()
    !mkdir -p {subset_dir}
    print(f'Created folder titled {subset_dir}')

    mask = data.obs['Sample ID'] == sample
    
    subset = data[mask]
    subset.write_h5ad(subset_h5ad)
    
    organ_id = ORGAN_MAPPING[tissue]

    print(f'Now running hra-workflows for {dataset_id, organ_id, sample}')
    run_all_hra_workflows(subset_h5ad, dataset_id, organ_id, subset_dir, queryLayersKey, use_singularity = False)


Created folder tlted data/gtex/GTEX-GTEX-1HSMQ-5011-SM-GKSJH


INFO /home/abueckle/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py 3.1.20250110105449


Now running hra-workflows for ('urn:gtex:GTEX-1HSMQ-5011-SM-GKSJH', 'UBERON:0000948', 'GTEX-1HSMQ-5011-SM-GKSJH')


WARNING Workflow checker warning:
https://raw.githubusercontent.com/hubmapconsortium/hra-workflows/main/steps/run-one.cwl:55:11: Source
                                                                                               'matrix_or_null' of
                                                                                               type ["null",
                                                                                               "File"] may be
                                                                                               incompatible
https://raw.githubusercontent.com/hubmapconsortium/hra-workflows/main/steps/run-one.cwl:61:7:    with sink
                                                                                                 'matrix' of type
                                                                                                 "File"
https://raw.githubusercontent.com/hubmapconsortium/hra-workflows/main/steps/run-one.cwl:65:11: So

{
    "directories": [
        {
            "class": "Directory",
            "basename": "azimuth",
            "listing": [
                {
                    "location": "file:///home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-1HSMQ-5011-SM-GKSJH/azimuth/summary.jsonld",
                    "basename": "summary.jsonld",
                    "class": "File",
                    "checksum": "sha1$e81f113882c5e4b57a3ca2be8cf00eb90b8cf448",
                    "size": 666956,
                    "path": "/home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-1HSMQ-5011-SM-GKSJH/azimuth/summary.jsonld"
                },
                {
                    "location": "file:///home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-1HSMQ-5011-SM-GKSJH/azimuth/annotations.csv",
                    "basename": "annotations.csv",
                    "class": "File",
                    "checksum": "sha1$2890bf5d7aac4511a60a03fcb930

...

INFO /home/abueckle/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py 3.1.20250110105449


Now running hra-workflows for ('urn:gtex:GTEX-15EOM-5003-SM-G64IH', 'UBERON:0002097', 'GTEX-15EOM-5003-SM-G64IH')


WARNING Workflow checker warning:
https://raw.githubusercontent.com/hubmapconsortium/hra-workflows/main/steps/run-one.cwl:65:11: Source
                                                                                               'matrix_with_crosswalking'
                                                                                               of type ["null",
                                                                                               "File"] may be
                                                                                               incompatible
https://raw.githubusercontent.com/hubmapconsortium/hra-workflows/main/steps/run-one.cwl:71:7:    with sink
                                                                                                 'matrix' of type
                                                                                                 "File"
                                                                                       

{
    "directories": [
        {
            "class": "Directory",
            "basename": "azimuth",
            "listing": [
                {
                    "location": "file:///home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-15EOM-5003-SM-G64IH/azimuth/report.json",
                    "basename": "report.json",
                    "class": "File",
                    "checksum": "sha1$8a65ec8f4503f0749abf63ab000d50b77e214d82",
                    "size": 567,
                    "path": "/home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-15EOM-5003-SM-G64IH/azimuth/report.json"
                }
            ],
            "location": "file:///home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-15EOM-5003-SM-G64IH/azimuth",
            "path": "/home/abueckle/Github/hra-pop-notebooks/annotations/data/gtex/GTEX-GTEX-15EOM-5003-SM-G64IH/azimuth"
        },
        {
            "class": "Directory",
            "base

INFO Final process status is success


# Transform outputs into CSV files